# IN HIS NAME
# NNDL - HW5 - Q2

# Import libraries we need

In [ ]:
from datasets import load_dataset, DatasetDict
import numpy as np
from transformers import AutoTokenizer
import re
import torch
import math
import time
from torch.optim import AdamW
import torch.nn.functional as F
from functools import partial
from torch.nn.utils import clip_grad_norm_
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, get_linear_schedule_with_warmup
import bitsandbytes as bnb
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import DataLoader
from typing import List, Dict, Any
from tqdm.auto import tqdm

# Load the dataset

In [ ]:
dataset = load_dataset("gretelai/synthetic_text_to_sql")
print(dataset)
print("Train size:", len(dataset["train"]))
print("Test size:", len(dataset["test"]))

# Reform the dataset

In [3]:
dataset = dataset.shuffle(seed=42)

train_valid_split = dataset["train"].train_test_split(
    test_size=300,  
    seed=42
)

new_train = train_valid_split["train"]
new_valid = train_valid_split["test"]


new_train = new_train.select(range(1000))

test_dataset = dataset["test"]

if len(test_dataset) > 300:
    test_dataset = test_dataset.shuffle(seed=42).select(range(300))

final_dataset = DatasetDict({
    "train": new_train,
    "validation": new_valid,
    "test": test_dataset
})

print(final_dataset)


DatasetDict({
    train: Dataset({
        features: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation'],
        num_rows: 300
    })
    test: Dataset({
        features: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation'],
        num_rows: 300
    })
})


In [4]:
train_df = final_dataset["train"].to_pandas()
valid_df = final_dataset["validation"].to_pandas()
test_df = final_dataset["test"].to_pandas()

print(len(train_df))
print(len(valid_df))
print(len(test_df))

1000
300
300


In [5]:
train_df.iloc[0]

id                                                                        71038
domain                                                                    space
domain_description            Space data on space exploration, satellite tec...
sql_complexity                                                      single join
sql_complexity_description          only one join (specify inner, outer, cross)
sql_task_type                                           analytics and reporting
sql_task_type_description     generating reports, dashboards, and analytical...
sql_prompt                    What is the latest launch date for each space ...
sql_context                   CREATE TABLE launches (id INT, agency_id INT, ...
sql                           SELECT a.name, MAX(l.launch_date) FROM launche...
sql_explanation               This query finds the latest launch date for ea...
Name: 0, dtype: object

# Compute length statistics

In [6]:
print(train_df.columns)

Index(['id', 'domain', 'domain_description', 'sql_complexity',
       'sql_complexity_description', 'sql_task_type',
       'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql',
       'sql_explanation'],
      dtype='str')


In [7]:
schema_lengths = [len(s) for s in train_df["sql_context"]]
question_lengths = [len(q) for q in train_df["sql_prompt"]]

schema_mean = np.mean(schema_lengths)
schema_std = np.std(schema_lengths)
schema_max = np.max(schema_lengths)

question_mean = np.mean(question_lengths)
question_std = np.std(question_lengths)
question_max = np.max(question_lengths)

print("Schema Length Statistics (Character Count)")
print("Mean:", round(schema_mean, 2))
print("Std Dev:", round(schema_std, 2))
print("Max:", round(schema_max,2))

print("\nQuestion Length Statistics (Character Count)")
print("Mean:", round(question_mean, 2))
print("Std Dev:", round(question_std, 2))
print("Max:", round(question_max,2))

Schema Length Statistics (Character Count)
Mean: 272.79
Std Dev: 138.68
Max: 1041

Question Length Statistics (Character Count)
Mean: 83.9
Std Dev: 26.75
Max: 197


In [8]:
schema_lengths_words = [len(s.split()) for s in train_df["sql_context"]]
question_lengths_words = [len(q.split()) for q in train_df["sql_prompt"]]

print("Schema Length (Word Count)")
print("Mean:", round(np.mean(schema_lengths_words), 2))
print("Std Dev:", round(np.std(schema_lengths_words), 2))
print("Max:", round(np.max(schema_lengths_words), 2))

print("\nQuestion Length (Word Count)")
print("Mean:", round(np.mean(question_lengths_words), 2))
print("Std Dev:", round(np.std(question_lengths_words), 2))
print("Max:", round(np.max(question_lengths_words), 2))


Schema Length (Word Count)
Mean: 32.44
Std Dev: 17.53
Max: 149

Question Length (Word Count)
Mean: 13.99
Std Dev: 4.27
Max: 32


# Simplify the schema

In [9]:
def simplify_schema_map(example):
    schema_text = example["sql_context"]
    
    simplified = []
    
    tables = re.findall(
        r"CREATE TABLE (\w+)\s*\((.*?)\);",
        schema_text,
        flags=re.DOTALL | re.IGNORECASE
    )
    
    for table_name, columns_text in tables:
        columns = [col.strip().split()[0] for col in columns_text.split(",")]
        simplified.append(f"{table_name}: " + ", ".join(columns))
    
    # Overwrite sql_context
    return {"sql_context": " | ".join(simplified)}


In [10]:
train_df_simp = final_dataset["train"].map(
    simplify_schema_map
)

valid_df_simp = final_dataset["validation"].map(
    simplify_schema_map
)

test_df_simp = final_dataset["test"].map(
    simplify_schema_map
)


In [11]:
# An example 
print("Original:")
print(train_df.iloc[0])
print("*"*60)
print("Simplified:")
train_df_simp[0]

Original:
id                                                                        71038
domain                                                                    space
domain_description            Space data on space exploration, satellite tec...
sql_complexity                                                      single join
sql_complexity_description          only one join (specify inner, outer, cross)
sql_task_type                                           analytics and reporting
sql_task_type_description     generating reports, dashboards, and analytical...
sql_prompt                    What is the latest launch date for each space ...
sql_context                   CREATE TABLE launches (id INT, agency_id INT, ...
sql                           SELECT a.name, MAX(l.launch_date) FROM launche...
sql_explanation               This query finds the latest launch date for ea...
Name: 0, dtype: object
************************************************************
Simplified:


{'id': 71038,
 'domain': 'space',
 'domain_description': 'Space data on space exploration, satellite technology, space debris mitigation, and astrobiology.',
 'sql_complexity': 'single join',
 'sql_complexity_description': 'only one join (specify inner, outer, cross)',
 'sql_task_type': 'analytics and reporting',
 'sql_task_type_description': 'generating reports, dashboards, and analytical insights',
 'sql_prompt': 'What is the latest launch date for each space agency?',
 'sql_context': 'launches: id, agency_id, launch_date | space_agencies: id, name',
 'sql': 'SELECT a.name, MAX(l.launch_date) FROM launches l JOIN space_agencies a ON l.agency_id = a.id GROUP BY a.name;',
 'sql_explanation': 'This query finds the latest launch date for each space agency by joining the launches and space_agencies tables on the agency_id column. It then groups the results by agency name and calculates the maximum launch date for each agency.'}

# Designing prompts

In [12]:
def make_chat_example(schema, question, gold_sql=None):
    messages = []
    
    system_msg = "You are an AI assistant that generates SQL queries only."
    messages.append({"role": "system", "content": system_msg})
    
    user_msg = f"Schema:\n{schema}\n\nQuestion:\n{question}"
    messages.append({"role": "user", "content": user_msg})
    
    if gold_sql is not None:
        messages.append({"role": "assistant", "content": gold_sql})
    
    return messages


In [13]:
def make_chat_example_no_answer(schema, question):
    messages = []
    
    system_msg = "You are an AI assistant that generates SQL queries only."
    messages.append({"role": "system", "content": system_msg})
    
    user_msg = f"Schema:\n{schema}\n\nQuestion:\n{question}"
    messages.append({"role": "user", "content": user_msg})
    
    messages.append({"role": "assistant", "content": ""})
    
    return messages


In [14]:
# A prompt instance with the answer
example = train_df_simp[0]  

chat_messages = make_chat_example(
    schema=example["sql_context"],
    question=example["sql_prompt"],
    gold_sql=example["sql"]  
)

for msg in chat_messages:
    print(msg)


{'role': 'system', 'content': 'You are an AI assistant that generates SQL queries only. Do not explain anything, do not write comments.'}
{'role': 'user', 'content': 'Schema:\nlaunches: id, agency_id, launch_date | space_agencies: id, name\n\nQuestion:\nWhat is the latest launch date for each space agency?'}
{'role': 'assistant', 'content': 'SELECT a.name, MAX(l.launch_date) FROM launches l JOIN space_agencies a ON l.agency_id = a.id GROUP BY a.name;'}


In [15]:
# A prompt instance without the answer
example = train_df_simp[0] 

chat_messages = make_chat_example_no_answer(
    schema=example["sql_context"],
    question=example["sql_prompt"]
)

for msg in chat_messages:
    print(msg)

{'role': 'system', 'content': 'You are an AI assistant that generates SQL queries only. Do not explain anything, do not write comments.'}
{'role': 'user', 'content': 'Schema:\nlaunches: id, agency_id, launch_date | space_agencies: id, name\n\nQuestion:\nWhat is the latest launch date for each space agency?'}
{'role': 'assistant', 'content': ''}


# SQL normalization

In [16]:
def normalize_sql(query: str) -> str:
    if query is None:
        return ""
    
    query = query.lower()
    query = query.replace('"', '').replace("'", "").replace("`", "")
    query = query.rstrip().rstrip(";")
    query = re.sub(r"\s+", " ", query)
    query = query.strip()
    
    return query



In [17]:
def raw_exact_match(prediction, reference):
    if prediction is None or reference is None:
        return False
    
    return prediction == reference


def normalized_exact_match(prediction, reference):
    norm_pred = normalize_sql(prediction)
    norm_ref = normalize_sql(reference)
    
    return norm_pred == norm_ref


In [18]:
pred = 'SELECT * FROM users ;'
ref  = "select * from users"

print("Raw EM:", raw_exact_match(pred, ref))
print("Normalized EM:", normalized_exact_match(pred, ref))


Raw EM: False
Normalized EM: True


# Load the model

In [19]:
model_name = "GSAI-ML/LLaDA-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
    use_cache=False
)

# Ensure pad token exists
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "<PAD>"})

if tokenizer.mask_token is None:
    tokenizer.add_special_tokens({"mask_token": "<MASK>"})



/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [20]:
mask_id = tokenizer.mask_token_id
eos_id = tokenizer.eos_token_id
pad_id = tokenizer.pad_token_id

print("Mask token id:", mask_id)
print("EOS token id:", eos_id)
print("PAD token id:", pad_id)

Mask token id: 126349
EOS token id: 126081
PAD token id: 126081


In [21]:
example = train_df_simp[0]
tokenized_example = tokenizer.apply_chat_template(
    make_chat_example(
        schema=example["sql_context"],
        question=example["sql_prompt"],
        gold_sql=example["sql"]
    ),
    max_length=512,
    truncation=True
)

print(tokenized_example)

[126080, 126346, 18621, 126347, 198, 198, 2496, 449, 289, 12458, 16841, 378, 27146, 12554, 20472, 1191, 13, 3488, 526, 8257, 4382, 11, 640, 526, 4223, 5635, 13, 126348, 126346, 3840, 126347, 198, 198, 17026, 25, 198, 6524, 46988, 25, 3414, 11, 10196, 2983, 11, 9688, 18517, 1221, 3450, 93382, 6824, 25, 3414, 11, 1717, 198, 198, 17188, 25, 198, 2372, 341, 268, 5950, 9688, 4347, 352, 1671, 3450, 10196, 30, 126348, 126346, 598, 10450, 126347, 198, 198, 17662, 259, 9150, 11, 19377, 7184, 106536, 18517, 8, 13539, 37768, 322, 47034, 3450, 93382, 6824, 259, 9809, 322, 100470, 2267, 2983, 373, 259, 9131, 58134, 15405, 259, 9150, 26, 126348, 126346, 598, 10450, 126347, 198, 198]


# Forward masking

In [22]:
def apply_chat_and_normalize(tokenizer, messages, max_length=512, truncation=True):
    tokout = tokenizer.apply_chat_template(messages, max_length=max_length, truncation=truncation)

    if isinstance(tokout, list) and all(isinstance(x, int) for x in tokout):
        input_ids = tokout
        attention_mask = [1] * len(input_ids)
        return {"input_ids": input_ids, "attention_mask": attention_mask}

    if isinstance(tokout, dict):
        if "input_ids" in tokout and "attention_mask" in tokout:
            return {"input_ids": tokout["input_ids"], "attention_mask": tokout["attention_mask"]}
        if "input_ids" in tokout and isinstance(tokout["input_ids"][0], list):
            return {"input_ids": tokout["input_ids"][0], "attention_mask": tokout["attention_mask"][0]}

    raise ValueError("Unexpected tokenizer.apply_chat_template output format: " + str(type(tokout)))


def tokenize_row_with_prompt_length(example, tokenizer, max_length=512):
    gold_sql = example.get("sql", None)
    full_msgs = make_chat_example(schema=example["sql_context"], question=example["sql_prompt"], gold_sql=gold_sql)
    prompt_only_msgs = make_chat_example_no_answer(schema=example["sql_context"], question=example["sql_prompt"])

    tok_prompt = apply_chat_and_normalize(tokenizer, prompt_only_msgs, max_length=max_length, truncation=True)
    prompt_len = len(tok_prompt["input_ids"])

    tok_full = apply_chat_and_normalize(tokenizer, full_msgs, max_length=max_length, truncation=True)

    return {
        "input_ids": tok_full["input_ids"],
        "attention_mask": tok_full["attention_mask"],
        "prompt_length": prompt_len
    }


In [23]:
map_fn = partial(tokenize_row_with_prompt_length, tokenizer=tokenizer, max_length=512)

train_tok = train_df_simp.map(map_fn, batched=False)
valid_tok = valid_df_simp.map(map_fn, batched=False)
test_tok  = test_df_simp.map(map_fn, batched=False)

In [24]:
def forward_mask_answer_only(input_ids_tensor, prompt_lengths, tokenizer, device=None):
    if device is None:
        device = input_ids_tensor.device
    input_ids = input_ids_tensor.clone().to(device)
    B, L = input_ids.shape
    prompt_lengths = torch.as_tensor(prompt_lengths, dtype=torch.long, device=device)

    mask_token_id = tokenizer.mask_token_id
    if mask_token_id is None:
        raise ValueError("Tokenizer has no mask_token_id.")

    answer_lengths = (L - prompt_lengths).clamp(min=0)
    p_mask = torch.rand(B, device=device)
    num_to_mask = torch.ceil(p_mask * answer_lengths.float()).to(torch.long)
    num_to_mask = torch.where(
        answer_lengths > 0,
        torch.clamp(min=1, max=L, input=num_to_mask),
        torch.zeros_like(num_to_mask),
    )

    noisy = input_ids.clone()
    masked_indices = torch.zeros_like(input_ids, dtype=torch.bool, device=device)

    for i in range(B):
        a_len = int(answer_lengths[i].item())
        if a_len <= 0:
            continue
        start = int(prompt_lengths[i].item())
        k = int(num_to_mask[i].item())
        perm = torch.randperm(a_len, device=device)[:k]
        chosen = (perm + start).to(torch.long)
        noisy[i, chosen] = mask_token_id
        masked_indices[i, chosen] = True

    return noisy, masked_indices, p_mask


In [25]:
def make_noisy_batch(input_ids, prompt_lengths, mask_token_id, tokenizer=None, device=None):
    if device is None:
        device = input_ids.device
    return forward_mask_answer_only(input_ids, prompt_lengths, tokenizer, device=device)


def compute_loss_with_masking(batch):
    input_ids = torch.as_tensor(batch["input_ids"], dtype=torch.long, device=device)
    attention_mask = torch.as_tensor(batch["attention_mask"], dtype=torch.long, device=device)
    prompt_length = torch.as_tensor(batch["prompt_length"], dtype=torch.long, device=device)

    noisy_input_ids, masked_indices, p_mask = make_noisy_batch(
        input_ids,
        prompt_length,
        tokenizer.mask_token_id,
        tokenizer=tokenizer,
        device=device,
    )

    outputs = model(
        input_ids=noisy_input_ids,
        attention_mask=attention_mask
    )
    logits = outputs.logits
    B, T, V = logits.shape

    logits_flat = logits.view(-1, V)
    targets_flat = input_ids.view(-1)
    masked_flat = masked_indices.view(-1)

    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_mask_flat = torch.ones_like(targets_flat, dtype=torch.bool, device=device)
    else:
        pad_mask_flat = targets_flat != pad_id

    final_mask_flat = masked_flat & pad_mask_flat

    if final_mask_flat.sum() == 0:
        return None

    token_loss_flat = F.cross_entropy(logits_flat, targets_flat, reduction="none")

    eps = 1e-6
    weights_per_example = 1.0 / (p_mask + eps)
    weights_tokens = (
        weights_per_example.unsqueeze(1)
        .expand(-1, T)
        .contiguous()
        .view(-1)
    )

    weighted_token_loss = token_loss_flat * final_mask_flat.float() * weights_tokens
    normalizer = (final_mask_flat.float() * weights_tokens).sum()

    loss = weighted_token_loss.sum() / (normalizer + 1e-12)

    return loss


# Training phase

In [26]:
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
# )

# model = AutoModelForCausalLM.from_pretrained(
#     "GSAI-ML/LLaDA-8B-Instruct",
#     quantization_config=bnb_config,
#     device_map="auto",
#     trust_remote_code=True,
# )

# model.config.use_cache = False

In [27]:
# lora_config = LoraConfig(
#     r=8,
#     lora_alpha=32,
#     lora_dropout=0.05,
#     target_modules=[
#         "q_proj",
#         "k_proj",
#         "v_proj",
#         "o_proj"
#     ],
#     bias="none",
#     task_type=TaskType.CAUSAL_LM,
# )

# model = get_peft_model(model, lora_config)
# model.print_trainable_parameters()

In [ ]:
MODEL_NAME = "GSAI-ML/LLaDA-8B-Instruct"
BATCH_SIZE = 8
NUM_EPOCHS = 3
LR = 2e-4
WEIGHT_DECAY = 0.0
MAX_GRAD_NORM = 1.0
WARMUP_STEPS = 100
MAX_STEPS = None
LOG_EVERY = 50
SAVE_EVERY = 1000
OUTPUT_DIR = "./llada_lora_ckpt"
MAX_LEN = 512

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    tokenizer
except NameError:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

_added_tokens = False
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "<pad>"})
    _added_tokens = True
if tokenizer.mask_token is None:
    tokenizer.add_special_tokens({"mask_token": "[MASK]"})
    _added_tokens = True

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

if _added_tokens:
    model.resize_token_embeddings(len(tokenizer))

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model.config.use_cache = False

def forward_mask_answer_only(input_ids_tensor: torch.LongTensor,
                             prompt_lengths: List[int],
                             tokenizer,
                             device=None):
    if device is None:
        device = input_ids_tensor.device
    input_ids = input_ids_tensor.clone().to(device)
    B, L = input_ids.shape
    prompt_lengths = torch.as_tensor(prompt_lengths, dtype=torch.long, device=device)
    mask_token_id = tokenizer.mask_token_id
    if mask_token_id is None:
        raise ValueError("Tokenizer has no mask_token_id.")

    answer_lengths = (L - prompt_lengths).clamp(min=0)
    p_mask = torch.rand(B, device=device)
    num_to_mask = torch.ceil(p_mask * answer_lengths.float()).to(torch.long)
    num_to_mask = torch.where(answer_lengths > 0, torch.clamp(num_to_mask, min=1), torch.zeros_like(num_to_mask))

    noisy = input_ids.clone()
    masked_indices = torch.zeros_like(input_ids, dtype=torch.bool, device=device)

    for i in range(B):
        a_len = int(answer_lengths[i].item())
        if a_len <= 0:
            continue
        start = int(prompt_lengths[i].item())
        k = int(num_to_mask[i].item())
        perm = torch.randperm(a_len, device=device)[:k]
        chosen = perm + start
        noisy[i, chosen] = mask_token_id
        masked_indices[i, chosen] = True

    return noisy, masked_indices, p_mask

def make_noisy_batch(input_ids, prompt_lengths, mask_token_id, tokenizer=tokenizer, device=None):
    return forward_mask_answer_only(input_ids, prompt_lengths, tokenizer, device=device)

def compute_loss_with_masking(batch: Dict[str, Any]):
    input_ids = torch.as_tensor(batch["input_ids"], dtype=torch.long, device=device)
    attention_mask = torch.as_tensor(batch["attention_mask"], dtype=torch.long, device=device)
    prompt_length = torch.as_tensor(batch["prompt_length"], dtype=torch.long, device=device)

    noisy_input_ids, masked_indices, p_mask = forward_mask_answer_only(input_ids, prompt_length.cpu().tolist(), tokenizer, device=device)

    outputs = model(input_ids=noisy_input_ids, attention_mask=attention_mask)
    logits = outputs.logits
    B, T, V = logits.shape

    logits_flat = logits.view(-1, V)
    targets_flat = input_ids.view(-1)
    masked_flat = masked_indices.view(-1)

    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_mask_flat = torch.ones_like(targets_flat, dtype=torch.bool, device=device)
    else:
        pad_mask_flat = targets_flat != pad_id

    final_mask_flat = masked_flat & pad_mask_flat
    if final_mask_flat.sum() == 0:
        return None

    token_loss_flat = F.cross_entropy(logits_flat, targets_flat, reduction="none")
    eps = 1e-6
    weights_per_example = 1.0 / (p_mask + eps)
    weights_tokens = weights_per_example.unsqueeze(1).expand(-1, T).contiguous().view(-1)
    weighted_token_loss = token_loss_flat * final_mask_flat.float() * weights_tokens
    normalizer = (final_mask_flat.float() * weights_tokens).sum()
    loss = weighted_token_loss.sum() / (normalizer + 1e-12)
    return loss

def collate_fn(batch: List[Dict[str, Any]]):
    input_ids_list = [torch.tensor(x["input_ids"], dtype=torch.long) for x in batch]
    attention_mask_list = [torch.tensor(x["attention_mask"], dtype=torch.long) for x in batch]
    prompt_lengths = torch.tensor([int(x["prompt_length"]) for x in batch], dtype=torch.long)

    max_len = max([t.size(0) for t in input_ids_list])
    pad_id = tokenizer.pad_token_id

    padded_ids = []
    padded_masks = []
    for ids, am in zip(input_ids_list, attention_mask_list):
        cur_len = ids.size(0)
        if cur_len < max_len:
            pad_len = max_len - cur_len
            ids = torch.cat([ids, torch.full((pad_len,), pad_id, dtype=torch.long)])
            am = torch.cat([am, torch.zeros(pad_len, dtype=torch.long)])
        padded_ids.append(ids)
        padded_masks.append(am)

    batch_input_ids = torch.stack(padded_ids, dim=0)
    batch_attention_mask = torch.stack(padded_masks, dim=0)

    return {"input_ids": batch_input_ids, "attention_mask": batch_attention_mask, "prompt_length": prompt_lengths}

train_loader = DataLoader(train_tok, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_tok, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

if MAX_STEPS is None:
    steps_per_epoch = math.ceil(len(train_loader))
    total_training_steps = NUM_EPOCHS * steps_per_epoch
else:
    total_training_steps = MAX_STEPS

scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_training_steps)

scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

global_step = 0
model.train()

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    epoch_steps = 0
    for step, batch in enumerate(train_loader):
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available(), dtype=torch.float16):
            loss = compute_loss_with_masking(batch)

        if loss is None:
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        scheduler.step()

        epoch_loss += loss.item()
        epoch_steps += 1
        global_step += 1

        if global_step % LOG_EVERY == 0:
            avg_loss = epoch_loss / max(1, epoch_steps)
            print(f"Epoch {epoch+1} Step {global_step} AvgLoss {avg_loss:.4f}")

        if global_step % SAVE_EVERY == 0:
            model.save_pretrained(f"{OUTPUT_DIR}/checkpoint-{global_step}")
            print(f"Saved checkpoint at step {global_step}")

    if epoch_steps > 0:
        print(f"Epoch {epoch+1} completed. Avg loss: {epoch_loss/epoch_steps:.4f}")

model.save_pretrained(OUTPUT_DIR)
print("Training completed. PEFT adapters saved to", OUTPUT_DIR)
